# Newton (XPBD) vs. FEniCSx (FEM) — Stage A on Colab

Hanging soft block: clamp the top face, let gravity stretch it, compare Newton's
fast XPBD solver against an accurate FEniCSx (dolfinx) FEM solve and analytic
bar theory.

**Runtime → Change runtime type → T4 GPU** before running.

## 1. Check the GPU

In [ ]:
!nvidia-smi

## 2. Install Newton + Warp

In [ ]:
# NVIDIA Warp (warp-lang) is Newton's core dependency and is pulled in
# automatically by this install -- no separate Warp step is needed.
!pip install -q "newton[examples]" matplotlib
# Fallback if the name/version misbehaves (installs warp-lang explicitly):
# !pip install -q warp-lang matplotlib git+https://github.com/newton-physics/newton.git
# Verify Warp is installed and sees the GPU:
import warp as wp; wp.init(); print('warp version:', wp.config.version, '| device:', wp.get_device())

## 3. Install FEniCSx (dolfinx) via fem-on-colab

`fem-on-colab` provides a Colab-compatible dolfinx + PETSc build. If this large
stack clashes with Newton/Warp in one runtime, run the FEM side in a **separate
notebook** — the two only exchange `data/*.npz` files.

In [ ]:
import os
# Colab-compatible dolfinx (real-mode PETSc) via fem-on-colab. NOTE: the URL is
# the 'release' build -- the old fenicsx-install-real.sh path is gone (404).
URL = "https://fem-on-colab.github.io/releases/fenicsx-install-release-real.sh"
!wget -q "{URL}" -O /tmp/fenicsx.sh
assert os.path.getsize("/tmp/fenicsx.sh") > 0, "installer download failed (0 bytes) -- check URL / Internet"
!bash /tmp/fenicsx.sh
import dolfinx, mpi4py, petsc4py
print("dolfinx", dolfinx.__version__)

## 4. Get the project files & install (editable)

Clone or upload the repo, `cd` into it, then `pip install -e .` so that
`common`, `newton_run`, `fenics_run`, `compare` import from anywhere — no
`sys.path` hacks needed.

In [ ]:
import os
REPO = 'https://github.com/AustrianTradingMachine/Newton.git'   # public repo
# Clone the repo into /content (Colab working dir). Re-runs reuse the checkout.
if not os.path.isdir('/content/Newton'):
    !git clone --depth 1 {REPO} /content/Newton
%cd /content/Newton
# Editable install so `common`, `fenics_run`, ... resolve anywhere:
!pip install -q -e .
print('cwd:', os.getcwd()); print(os.listdir('.'))

## 5. Run Newton (XPBD)

In [ ]:
!python -m newton_run.run_stage_a

## 6. Run FEniCSx (FEM)

In [ ]:
# tet: Newton's shared mesh (node-for-node);  hex: independent mesh (element-effect reference)
!python -m fenics_run.run_stage_a --element tet
!python -m fenics_run.run_stage_a --element hex

## 7. Compare + show figures

In [ ]:
!python -m compare.metrics
from IPython.display import Image, display
for f in ['stage_a_profile.png', 'stage_a_settling.png', 'stage_a_error_hist.png']:
    display(Image('figures/' + f))

## 8. Stage B — rigid-sphere contact (dolfinx penalty, no C++)

Quasi-static, frictionless indentation with a custom penalty contact written
entirely in UFL/Python. Runs the variants in `STAGEB_VARIANTS` — a **tet
penalty-strength sweep (kn×5 vs kn×50)** plus a **hex** reference — and overlays
the contact force vs. indentation with a Hertz anchor, so both the penalty
effect and the element-locking effect are visible.

In [ ]:
!python -m fenics_run.run_stage_b
from IPython.display import Image, display
for f in ['stage_b_force.png', 'stage_b_profile.png']:
    display(Image('figures/' + f))

## 9. Stage B — Newton (XPBD) contact + FEM comparison

Drives the same indentation in Newton's XPBD solver (kinematic sphere, clamped
slab) and overlays the deformed dimple against the FEM variants. XPBD exposes no
calibrated contact force, so the comparison axis is deformation / penetration —
precisely the difference between a fast positional solver and implicit FEM.

In [ ]:
# Newton XPBD indentation (CUDA), then overlay the deformed dimple vs FEM
!python -m newton_run.run_stage_b
!python -m compare.stage_b
from IPython.display import Image, display
for f in ['stage_b_compare_profile.png', 'stage_b_newton_penetration.png']:
    display(Image('figures/' + f))

## 10. Evaluation

The runs above wrote all `data/*.npz`. For the full quantitative evaluation —
deflection, **internal / contact / kinetic energy**, force and dimple comparisons
— open the dedicated analysis notebooks:

* **`10_stage_a_analysis.ipynb`** (incl. FEM-vs-analytic validity + both Newton solvers)
* **`20_stage_b_analysis.ipynb`**
* **`30_convergence_analysis.ipynb`** (run §14 first)
* **`40_friction_analysis.ipynb`** (run §14 first — Coulomb friction force / energy)

## 11. (Optional) Newton's superpower — differentiable stiffness fit

Backpropagate through the Newton simulation to fit the effective material
stiffness to the FEM reference. The fitted `theta*` quantifies how much softer /
stiffer Newton's effective material is. Then re-run §9 of
`10_stage_a_analysis.ipynb` to plot it.

In [ ]:
# Newton's superpower: backprop through the sim to fit effective stiffness vs FEM.
# Heavier (full trajectory + autodiff, SemiImplicit solver); the lr may need tuning.
!python -m newton_run.diffsim_stage_a

## 12. (Optional) Dynamic drop — the literal example, Newton vs FEM

The original `rigid_soft_contact` scenario: a rigid sphere **dropped** onto a soft
block resting on the ground (dynamic impact, gravity on). Newton XPBD vs FEM
**Newmark elastodynamics + penalty contact** (with ~10% Kelvin-Voigt damping).
Compares the transient: sphere trajectory, penetration, block strain & kinetic
energy, and the FEM contact force. Heavier; the FEM drop's `dt` / damping may need
tuning.

In [ ]:
!python -m newton_run.example_rigid_soft_contact   # literal XPBD drop (CUDA)
!python -m fenics_run.run_drop                      # FEM Newmark drop (needs dolfinx)
!python -m compare.drop
import os
from IPython.display import Image, display
for f in ['drop_sphere_z.png', 'drop_penetration.png', 'drop_strain_energy.png',
          'drop_kinetic_energy.png', 'drop_contact_force.png']:
    if os.path.exists('figures/' + f):
        display(Image('figures/' + f))

## 13. (Optional) Effective stress-strain — material fidelity

Confined uniaxial strain `F = diag(1,1,λ)`: axial stress vs. stretch, FEM and
Newton vs. the analytic Neo-Hookean — into the **large-strain** regime where Stage
A's small-strain material match no longer holds.

In [ ]:
!python -m fenics_run.run_stress_strain   # FEM uniaxial sweep (needs dolfinx)
!python -m newton_run.run_stress_strain   # Newton SemiImplicit uniaxial sweep (CUDA)
!python -m compare.stress_strain
import os
from IPython.display import Image, display
if os.path.exists('figures/stress_strain.png'):
    display(Image('figures/stress_strain.png'))

## 14. Explicit Stage A solver, (4) convergence study & friction

The second Newton solver (explicit / SemiImplicit) for Stage A, the convergence
study (XPBD iters/substeps + FEM h-refinement) and the Coulomb friction scenario.
Analysis lives in `30_convergence_analysis.ipynb` and `40_friction_analysis.ipynb`.

In [ ]:
# Explicit Stage A solver (the second Newton solver to validate vs FEM + analytic)
!python -m newton_run.run_stage_a --solver semi_implicit
# (4) Convergence study
!python -m newton_run.convergence_stage_a
!python -m fenics_run.convergence_stage_a
!python -m compare.convergence
# Friction -- sliding block, Coulomb
!python -m fenics_run.run_friction
!python -m newton_run.run_friction
!python -m compare.friction
import os
from IPython.display import Image, display
for f in ['convergence_newton.png', 'convergence_fem.png',
          'friction_force.png', 'friction_work.png', 'friction_slip.png']:
    if os.path.exists('figures/' + f):
        display(Image('figures/' + f))

## 15. Run everything & summarize logs

One cell that runs every stage, captures each stage's output to
`logs/<stage>.log`, and prints a compact **OK / ERR summary** with the key result
lines (or the traceback tail on failure) — so you don't have to hunt per-cell
outputs. The full summary is also written to `logs/summary.txt`.

Heavy (runs the whole pipeline) — comment out stages you don't need. Requires
§2–§4 (Newton + dolfinx installed, repo cloned) to have run first.

In [ ]:
import subprocess, os, re
os.chdir('/content/Newton')
os.makedirs('logs', exist_ok=True)

STAGES = [
    ("Newton Stage A (XPBD)",     "python -m newton_run.run_stage_a"),
    ("Newton Stage A (explicit)", "python -m newton_run.run_stage_a --solver semi_implicit"),
    ("FEM Stage A tet",           "python -m fenics_run.run_stage_a --element tet"),
    ("FEM Stage A hex",           "python -m fenics_run.run_stage_a --element hex"),
    ("Compare Stage A",           "python -m compare.metrics"),
    ("FEM Stage B",               "python -m fenics_run.run_stage_b"),
    ("Newton Stage B",            "python -m newton_run.run_stage_b"),
    ("Compare Stage B",           "python -m compare.stage_b"),
    ("Newton convergence",        "python -m newton_run.convergence_stage_a"),
    ("FEM convergence",           "python -m fenics_run.convergence_stage_a"),
    ("Compare convergence",       "python -m compare.convergence"),
    ("FEM friction",              "python -m fenics_run.run_friction"),
    ("Newton friction",           "python -m newton_run.run_friction"),
    ("Compare friction",          "python -m compare.friction"),
    ("Diffsim theta*",            "python -m newton_run.diffsim_stage_a"),
    ("Drop Newton",               "python -m newton_run.example_rigid_soft_contact"),
    ("Drop FEM",                  "python -m fenics_run.run_drop"),
    ("Compare drop",              "python -m compare.drop"),
    ("Stress-strain FEM",         "python -m fenics_run.run_stress_strain"),
    ("Stress-strain Newton",      "python -m newton_run.run_stress_strain"),
    ("Compare stress-strain",     "python -m compare.stress_strain"),
]
KEY = re.compile(r"(wrote |tip vertical drop|tip ratio|max .*displacement|residual RMS|"
                 r"max rel|plateau|theta\*|deviation|F=|pen=|settled at frame|dolfinx )", re.I)

rows = []
for label, cmd in STAGES:
    slug = re.sub(r"\W+", "_", label).strip("_").lower()
    logf = f"logs/{slug}.log"
    with open(logf, "w") as fh:
        rc = subprocess.run(cmd, shell=True, stdout=fh, stderr=subprocess.STDOUT).returncode
    lines = open(logf).read().splitlines()
    tag = "OK " if rc == 0 else "ERR"
    hits = ([l.strip() for l in lines if KEY.search(l)][-3:] if rc == 0
            else [l for l in lines if l.strip()][-8:])   # traceback tail on failure
    rows.append((tag, label, hits))
    print(f"[{tag}] {label}")

report = []
for tag, label, hits in rows:
    report.append(f"[{tag}] {label}")
    report += [f"      {h}" for h in hits]
text = "\n".join(report)
open("logs/summary.txt", "w").write(text + "\n")
print("\n==================== SUMMARY ====================")
print(text)
print("\n(per-stage logs in logs/, full summary in logs/summary.txt)")